### Project Goal
#### Load Data → Clean Data → Run 4 Forecast Models → Create Separate Charts → Create Integrated Comparison Chart → Export for Portfolio

### Cell 1 — Project setup

In [ ]:
import sys
from pathlib import Path

# Resolve the project root whether this notebook is run from the repo
# root or from inside the notebooks/ folder (portable across machines).
PROJECT_PATH = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROJECT_PATH = str(PROJECT_PATH)

if PROJECT_PATH not in sys.path:
    sys.path.append(PROJECT_PATH)

print("Project path added successfully:")
print(PROJECT_PATH)

### Cell 2 — Reload latest bot files

In [ ]:
import importlib

import bots.ingestion_bot
import bots.petrol_transform_bot
import bots.petrol_linear_forecast_bot
import bots.petrol_arima_forecast_bot
import bots.petrol_lstm_forecast_bot
import bots.petrol_autokeras_forecast_bot
import bots.petrol_visualization_bot

importlib.reload(bots.ingestion_bot)
importlib.reload(bots.petrol_transform_bot)
importlib.reload(bots.petrol_linear_forecast_bot)
importlib.reload(bots.petrol_arima_forecast_bot)
importlib.reload(bots.petrol_lstm_forecast_bot)
importlib.reload(bots.petrol_autokeras_forecast_bot)
importlib.reload(bots.petrol_visualization_bot)

print("All petrol bot files reloaded successfully.")

### Cell 3 — Import the bots

In [ ]:
from bots.ingestion_bot import IngestionBot
from bots.petrol_transform_bot import PetrolTransformBot
from bots.petrol_linear_forecast_bot import PetrolLinearForecastBot
from bots.petrol_arima_forecast_bot import PetrolARIMAForecastBot
from bots.petrol_lstm_forecast_bot import PetrolLSTMForecastBot
from bots.petrol_autokeras_forecast_bot import PetrolAutoKerasForecastBot
from bots.petrol_visualization_bot import PetrolVisualizationBot

print("All petrol bots imported successfully.")

### Cell 4 — Load raw petrol dataset

In [ ]:
from pathlib import Path

ingestion = IngestionBot()

DATA_PATH = Path(PROJECT_PATH) / "data" / "incoming" / "GES_Deutschland_2023-2026.csv"

df_raw = ingestion.load_file(str(DATA_PATH))

display(df_raw.head())
print("Raw dataset shape:", df_raw.shape)

### Cell 5 — Clean petrol dataset

In [ ]:
petrol_transform = PetrolTransformBot()

df_petrol = petrol_transform.clean_petrol_data(df_raw)

display(df_petrol.head())
print("Cleaned dataset shape:", df_petrol.shape)

### Cell 6 — Check cleaned data

In [ ]:
print("Dataset shape:")
print(df_petrol.shape)

print("\nMissing values:")
print(df_petrol.isnull().sum())

print("\nData types:")
print(df_petrol.dtypes)

print("\nFirst rows:")
display(df_petrol.head())

print("\nLast rows:")
display(df_petrol.tail())

### Cell 7 — Run Linear Regression forecast

In [ ]:
linear_bot = PetrolLinearForecastBot()

forecast_df = linear_bot.forecast_next_days(
    df=df_petrol,
    date_col="date",
    value_col="price",
    days=30,
    lags=7
)

forecast_df["model"] = "Linear Regression"

display(forecast_df.head())
display(forecast_df.tail())

### Cell 8 — Run ARIMA forecast

In [ ]:
arima_bot = PetrolARIMAForecastBot()

arima_forecast_df = arima_bot.forecast_next_days(
    df=df_petrol,
    date_col="date",
    value_col="price",
    days=30,
    order=(1, 1, 1)
)

arima_forecast_df["model"] = "ARIMA"

display(arima_forecast_df.head())
display(arima_forecast_df.tail())

 ### Cell 9 — Run LSTM forecast

In [ ]:
lstm_bot = PetrolLSTMForecastBot()

lstm_forecast_df = lstm_bot.forecast_next_days(
    df=df_petrol,
    date_col="date",
    value_col="price",
    days=30,
    lookback=30,
    epochs=50,
    batch_size=16,
    verbose=0
)

lstm_forecast_df["model"] = "LSTM"

display(lstm_forecast_df.head())
display(lstm_forecast_df.tail())

### Cell 10 — Run AutoKeras forecast

In [ ]:
autokeras_bot = PetrolAutoKerasForecastBot()

autokeras_forecast_df = autokeras_bot.forecast_next_days(
    df=df_petrol,
    date_col="date",
    value_col="price",
    days=30,
    lags=14,
    max_trials=5,
    epochs=30
)

autokeras_forecast_df["model"] = "AutoKeras"

display(autokeras_forecast_df.head())
display(autokeras_forecast_df.tail())

### Cell 11 — Combine all forecasts

In [ ]:
import pandas as pd

all_forecasts_df = pd.concat(
    [
        forecast_df,
        arima_forecast_df,
        lstm_forecast_df,
        autokeras_forecast_df
    ],
    ignore_index=True
)

display(all_forecasts_df.head())
display(all_forecasts_df.tail())

print(all_forecasts_df["model"].value_counts())

### Cell 12 — Create visualization bot

In [ ]:
petrol_viz = PetrolVisualizationBot()

### Cell 13 — Linear Regression chart

In [ ]:
fig_linear = petrol_viz.single_model_forecast_chart(
    actual_df=df_petrol,
    forecast_df=forecast_df,
    date_col="date",
    value_col="price",
    forecast_col="forecast",
    model_name="Linear Regression"
)

### Cell 14 — ARIMA chart

In [ ]:
fig_arima = petrol_viz.single_model_forecast_chart(
    actual_df=df_petrol,
    forecast_df=arima_forecast_df,
    date_col="date",
    value_col="price",
    forecast_col="forecast",
    model_name="ARIMA"
)

### Cell 15 — LSTM chart

In [ ]:
fig_lstm = petrol_viz.single_model_forecast_chart(
    actual_df=df_petrol,
    forecast_df=lstm_forecast_df,
    date_col="date",
    value_col="price",
    forecast_col="forecast",
    model_name="LSTM"
)

### Cell 16 — AutoKeras chart

In [ ]:
fig_autokeras = petrol_viz.single_model_forecast_chart(
    actual_df=df_petrol,
    forecast_df=autokeras_forecast_df,
    date_col="date",
    value_col="price",
    forecast_col="forecast",
    model_name="AutoKeras"
)

### Cell 17 — Integrated forecast comparison chart

In [ ]:
fig_integrated = petrol_viz.integrated_forecast_comparison_chart(
    actual_df=df_petrol,
    all_forecasts_df=all_forecasts_df,
    date_col="date",
    value_col="price",
    forecast_col="forecast",
    model_col="model"
)

### Cell 18 — Export charts for the portfolio website

In [ ]:
from pathlib import Path

CHARTS_DIR = Path(PROJECT_PATH) / "charts"
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

fig_linear.write_html(CHARTS_DIR / "petrol_linear_forecast.html")
fig_arima.write_html(CHARTS_DIR / "petrol_arima_forecast.html")
fig_lstm.write_html(CHARTS_DIR / "petrol_lstm_forecast.html")
fig_autokeras.write_html(CHARTS_DIR / "petrol_autokeras_forecast.html")
fig_integrated.write_html(CHARTS_DIR / "petrol_integrated_forecast_comparison.html")

print("All 5 charts exported to:", CHARTS_DIR)

### Summary

This notebook trains four forecasting approaches (Linear Regression, ARIMA, LSTM, and AutoKeras) on Germany's daily E10 petrol price (2023–2026), compares them in a single integrated chart, and exports all five charts as standalone interactive HTML files used on the Petrol Forecasting case study page on gimeno.tech.

As noted in the case study writeup: treat these as scenario bands for planning rather than a single-point forecast, and re-run this pipeline periodically as new daily price data becomes available.

### Pareto analysis (80/20 rule) — where does the price movement concentrate?

Rather than applying Pareto to a revenue breakdown (this is a price forecasting project, not
a sales one), the natural 80/20 question here is: **do a small number of trading days
account for most of the total price movement?** Each day's absolute price change
(`|price[t] - price[t-1]|`) is ranked, then plotted against its cumulative share of the total
movement across the full 2023–2026 series.

In [ ]:
import plotly.graph_objects as go

df_moves = df_petrol.copy()
df_moves["abs_move"] = df_moves["price"].diff().abs()
df_moves = df_moves.dropna(subset=["abs_move"])

total_move = df_moves["abs_move"].sum()
ranked = df_moves[["date", "abs_move"]].sort_values("abs_move", ascending=False).reset_index(drop=True)
ranked["cum_pct"] = ranked["abs_move"].cumsum() / total_move * 100

n_days = len(ranked)
n_top20 = round(n_days * 0.2)
pct_move_top20 = ranked["abs_move"].iloc[:n_top20].sum() / total_move * 100
n_for_80 = (ranked["cum_pct"] <= 80).sum() + 1
print(f"Total cumulative |price change|, 2023-2026: {total_move:.2f} EUR over {n_days} days")
print(f"Top 20% of days ({n_top20} of {n_days}) account for {pct_move_top20:.1f}% of total movement")
print(f"{n_for_80} of {n_days} days ({n_for_80 / n_days * 100:.1f}%) are needed to reach 80% of movement")

top50 = ranked.head(50)
fig = go.Figure()
fig.add_trace(go.Bar(
    x=top50["date"].dt.strftime("%Y-%m-%d"),
    y=top50["abs_move"],
    name="Daily |Δ price| (EUR)",
    marker_color="#5B8FF9"
))
fig.add_trace(go.Scatter(
    x=top50["date"].dt.strftime("%Y-%m-%d"),
    y=top50["cum_pct"],
    name="Cumulative % of total movement",
    yaxis="y2",
    mode="lines+markers",
    line=dict(color="#F6BD16", width=2)
))

fig.update_layout(
    title={"text": "Pareto Analysis — Top 50 Highest-Movement Days (E10 Price)", "x": 0.05, "font": {"size": 20, "color": "white"}},
    paper_bgcolor="#0b0f19",
    plot_bgcolor="#0b0f19",
    font={"family": "Arial", "color": "white", "size": 12},
    xaxis=dict(tickangle=-60, showgrid=True, gridcolor="#22304a"),
    yaxis=dict(title="Daily price move (EUR)", showgrid=True, gridcolor="#22304a"),
    yaxis2=dict(title="Cumulative % of total movement", overlaying="y", side="right", range=[0, 100]),
    margin=dict(l=60, r=60, t=70, b=140),
    height=620, width=1050
)

fig.write_html(CHARTS_DIR / "petrol_pareto_movement_days.html", include_plotlyjs=True,
    full_html=True, default_width=1050, default_height=620)
fig.show()

**Finding: moderate concentration, not a strict 80/20 split.** The top 20% of trading days
(242 of 1,208) account for **54.2%** of the total cumulative price movement over 2023–2026,
and it takes **44.4%** of all days to reach 80% of total movement. That's meaningfully more
concentrated than a uniform distribution (where 20% of days would account for only 20% of
movement), but it falls short of a textbook 80/20 pattern. **Business implication:** a
relatively compact set of high-volatility days (clustered around specific events, e.g. late
Feb 2023 and early March 2026) drives a disproportionate share of total price movement —
these are the periods where forecast error is likeliest to spike, and where hedging /
re-forecasting attention is most valuable, rather than treating every day as equally
unpredictable.